# Stable classification losses

**P1 Core · D3 Synthesis · 100 minutes**

In [ ]:
import numpy as np
from scipy.special import logsumexp
from pathlib import Path

def locate(relative: str, local_name: str | None = None) -> Path:
    candidates = []
    if local_name:
        candidates.append(Path.cwd() / local_name)
    candidates.extend(root / relative for root in [Path.cwd(), *Path.cwd().parents])
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"Cannot find {relative}. Run from the course clone or place the downloaded data beside this notebook."
    )


## Task

Implement stable row-wise softmax and mean cross-entropy from integer labels. Verify shift invariance and extreme logits.

In [ ]:
def stable_softmax(logits: np.ndarray) -> np.ndarray:
    shifted = logits - logits.max(axis=1, keepdims=True)
    exp = np.exp(shifted)
    return exp / exp.sum(axis=1, keepdims=True)

def cross_entropy(logits: np.ndarray, labels: np.ndarray) -> float:
    log_normaliser = logsumexp(logits, axis=1)
    return float(np.mean(log_normaliser - logits[np.arange(len(labels)), labels]))

In [ ]:
logits = np.array([[1000., 1001., 999.], [-1000., -1001., -999.]])
labels = np.array([1, 2])
probability = stable_softmax(logits)
loss = cross_entropy(logits, labels)
assert np.isfinite(probability).all() and np.isfinite(loss)
assert np.allclose(probability.sum(axis=1), 1.0)
assert np.allclose(probability, stable_softmax(logits + 12345.))
assert loss < 1.0
probability, loss

## Dry Bean guided fit

Load the teaching CSV, stratify a subset, scale inside a pipeline, fit multinomial logistic regression, and report log loss plus per-class confusion. Class labels must not enter preprocessing features.

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
path = locate('datasets/teaching/dry-bean/observations.csv', 'observations.csv')
beans = pd.read_csv(path).groupby('Class', group_keys=False).sample(n=300, random_state=7)
X_train, X_test, y_train, y_test = train_test_split(
    beans.drop(columns='Class'), beans['Class'], test_size=0.25, random_state=7, stratify=beans['Class'])
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train, y_train)
loss_library = log_loss(y_test, model.predict_proba(X_test), labels=model.classes_)
assert loss_library < 0.6
loss_library

## Transfer

On an imbalanced three-class oracle, compare macro-F1, per-class recall and log loss; explain how loss can change without changing argmax decisions.